# Ray Serve Runner (Colab GPU)

Run this notebook on a GPU runtime to launch a Ray Serve OpenAI-compatible endpoint and export a public URL for local benchmarking.

In [5]:
import os, re, sys, time, textwrap, subprocess
from pathlib import Path
from urllib import request

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ray[serve]', 'transformers', 'accelerate', 'pyngrok'])
print('Installed: ray[serve], transformers, accelerate, pyngrok')

Installed: ray[serve], transformers, accelerate, pyngrok


In [6]:
RAY_SERVE_MODEL = os.environ.get('RAY_SERVE_MODEL', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0')
RAY_SERVE_PORT = int(os.environ.get('RAY_SERVE_PORT', '8001'))
RAY_SERVE_LOG_PATH = '/tmp/ray_serve_openai.log'
RAY_SERVE_APP_PATH = '/tmp/ray_serve_openai_app.py'

if 'ray_serve_proc' in globals() and ray_serve_proc is not None and ray_serve_proc.poll() is None:
    ray_serve_proc.terminate()
    time.sleep(2)
if 'ray_serve_cfd' in globals() and ray_serve_cfd is not None and ray_serve_cfd.poll() is None:
    ray_serve_cfd.terminate()
    time.sleep(1)

app_code = f'''
import os
import time
from typing import Any

import ray
from fastapi import FastAPI
from ray import serve
from transformers import pipeline

MODEL_NAME = os.environ.get('RAY_SERVE_MODEL', '{RAY_SERVE_MODEL}')
PORT = int(os.environ.get('RAY_SERVE_PORT', '{RAY_SERVE_PORT}'))

api = FastAPI()
generator = pipeline('text-generation', model=MODEL_NAME, device_map='auto')

@serve.deployment(ray_actor_options={{'num_gpus': 1}})
@serve.ingress(api)
class OpenAICompat:
    @api.get('/v1/models')
    def models(self) -> dict[str, Any]:
        return {{'object': 'list', 'data': [{{'id': MODEL_NAME, 'object': 'model'}}]}}

    @api.post('/v1/chat/completions')
    def chat_completions(self, payload: dict[str, Any]) -> dict[str, Any]:
        messages = payload.get('messages') or []
        prompt = '\\n'.join(m.get('content', '') for m in messages if isinstance(m, dict))
        max_tokens = int(payload.get('max_tokens', 64))
        out = generator(prompt or 'hello', max_new_tokens=max_tokens)[0]['generated_text']
        text = out[len(prompt):].strip() if out.startswith(prompt) else out
        return {{
            'id': 'chatcmpl-' + str(int(time.time())),
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': MODEL_NAME,
            'choices': [{{
                'index': 0,
                'message': {{'role': 'assistant', 'content': text}},
                'finish_reason': 'stop',
            }}],
        }}

if __name__ == '__main__':
    ray.init(ignore_reinit_error=True)
    serve.start(http_options={{'host': '0.0.0.0', 'port': PORT}})
    serve.run(OpenAICompat.bind(), name='mws-openai', route_prefix='/')
    print('RAY_SERVE_READY model=' + MODEL_NAME + ' port=' + str(PORT), flush=True)
    while True:
        time.sleep(30)
'''

Path(RAY_SERVE_APP_PATH).write_text(textwrap.dedent(app_code).strip() + '\n')
os.environ['RAY_SERVE_MODEL'] = RAY_SERVE_MODEL
os.environ['RAY_SERVE_PORT'] = str(RAY_SERVE_PORT)

ray_serve_proc = subprocess.Popen(
    [sys.executable, RAY_SERVE_APP_PATH],
    stdout=open(RAY_SERVE_LOG_PATH, 'w'),
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(25)

# Confirm local endpoint before exposing tunnel.
local_ok = False
for _ in range(20):
    if ray_serve_proc.poll() is not None:
        break
    try:
        with request.urlopen(request.Request(f'http://127.0.0.1:{RAY_SERVE_PORT}/v1/models', method='GET'), timeout=5) as resp:
            print('Local health HTTP', resp.status)
            print(resp.read(200).decode('utf-8', 'replace'))
            local_ok = True
            break
    except Exception as exc:
        print('Local health retry:', type(exc).__name__, exc)
        time.sleep(2)

if not local_ok:
    print('Ray Serve local endpoint did not become healthy. Log tail:')
    if Path(RAY_SERVE_LOG_PATH).exists():
        for ln in Path(RAY_SERVE_LOG_PATH).read_text().splitlines()[-60:]:
            print(ln)

PUBLIC_BASE_URL = None
try:
    from pyngrok import ngrok

    if os.environ.get('NGROK_AUTHTOKEN'):
        ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
    PUBLIC_BASE_URL = ngrok.connect(RAY_SERVE_PORT, 'http').public_url.rstrip('/')
except Exception as exc:
    print('ngrok unavailable; falling back to cloudflared:', exc)
    subprocess.run(
        [
            'bash',
            '-lc',
            "which cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)",
        ],
        check=True,
    )
    ray_serve_cfd = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{RAY_SERVE_PORT}', '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    pat = re.compile(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com')
    t0 = time.time()
    seen = []
    while time.time() - t0 < 40:
        line = ray_serve_cfd.stdout.readline() if ray_serve_cfd.stdout else ''
        if not line:
            time.sleep(0.2)
            continue
        seen.append(line.rstrip())
        m = pat.search(line)
        if m:
            PUBLIC_BASE_URL = m.group(0).strip().rstrip('/')
            break
    if PUBLIC_BASE_URL is None:
        print('cloudflared logs tail:')
        for ln in seen[-20:]:
            print(ln)

print('PUBLIC_BASE_URL=', PUBLIC_BASE_URL)

if not PUBLIC_BASE_URL:
    raise RuntimeError('Failed to obtain public base URL from ngrok/cloudflared')

health_ok = False
for _ in range(20):
    try:
        with request.urlopen(request.Request(PUBLIC_BASE_URL + '/v1/models', method='GET'), timeout=8) as resp:
            print('Public health HTTP', resp.status)
            print(resp.read(300).decode('utf-8', 'replace'))
            health_ok = True
            break
    except Exception as exc:
        print('Public health retry:', type(exc).__name__, exc)
        time.sleep(2)

if not health_ok:
    print('Public endpoint did not become healthy. Check tunnel and log tail:')
    if Path(RAY_SERVE_LOG_PATH).exists():
        for ln in Path(RAY_SERVE_LOG_PATH).read_text().splitlines()[-60:]:
            print(ln)

Local health HTTP 200
{"object":"list","data":[{"id":"sshleifer/tiny-gpt2","object":"model"}]}


ERROR:pyngrok.process.ngrok:t=2026-03-23T17:35:34+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-23T17:35:34+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


ngrok unavailable; falling back to cloudflared: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
PUBLIC_BASE_URL= https://proof-sends-cakes-exist.trycloudflare.com
Public health retry: URLError <urlopen error [Errno -2] Name or service not known>
Public health retry: URLError <urlopen error [Errno -2] Name or service not known>
Public health retry: URLError <urlopen error [Errno -2] Name or service not known>
Public health retry: URLError <urlopen error [Errno -2] Name or service not known>
Public health retry: URLError <urlopen error [Errno -2] Name or service not known>
Public health retry: URLError <urlopen error [Errno -2] Name or service not known>
Public health retry: URLError <urlopen error [Errno -2] Name or service not known>
Public health retr

In [7]:
print('Local machine commands:')
print(
    'uv run python scripts/cloud/write_remote_config.py '
    f'--backend ray-serve --base-url {PUBLIC_BASE_URL} --r5 '
    f"--streaming-model '{RAY_SERVE_MODEL}' --agentic-model '{RAY_SERVE_MODEL}' "
    '--output configs/live_ray_serve_remote_r5.json'
)
print('uv run python scripts/probe_live_endpoints.py --config configs/live_ray_serve_remote_r5.json')
print('uv run mws-bench sweep --config configs/live_ray_serve_remote_r5.json --output results/live_ray_serve_remote_sweep_r5.csv --trace-output-dir results/traces/live_ray_serve_remote_r5')

Local machine commands:
uv run python scripts/cloud/write_remote_config.py --backend ray-serve --base-url https://proof-sends-cakes-exist.trycloudflare.com --r5 --streaming-model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' --agentic-model 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' --output configs/live_ray_serve_remote_r5.json
uv run python scripts/probe_live_endpoints.py --config configs/live_ray_serve_remote_r5.json
uv run mws-bench sweep --config configs/live_ray_serve_remote_r5.json --output results/live_ray_serve_remote_sweep_r5.csv --trace-output-dir results/traces/live_ray_serve_remote_r5
